### Needs `01-Prep-BaselinePipelineA.ipynb` to be executed first

Reads from:
- `Baseline-PipelineA_ynorm.json`

# Preperations

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Import Data

In [2]:
answers = pd.read_json("Baseline-PipelineA_ynorm.json")

CATEGORIES = ["value", "unit"]
VARIANTS = ["Base", "A"]

#### Cleanup Import

In [3]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)

for col in pipeline_cols:
    answers[col] = answers[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [4]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def check_hit_exactly(row, extraction_col, gs_col) -> bool:
    # Preparing gold-standard value(s) for comparison
    gs_val  = row[gs_col]
    if isinstance(gs_val,  list):
        gs_set = set(gs_val)
    elif not pd.isna(gs_val):
        gs_set = {gs_val}
    else:
        gs_set = set()

    # Preparing extracted value(s) for comparison
    ext_val = row[extraction_col]
    if isinstance(ext_val, list):
        ext_set = set(ext_val)
    elif not pd.isna(ext_val):
        ext_set = {ext_val}
    else:
        ext_set = set()

    # Compare value(s), as there could be more than one (e.g., Allianz) or just one (mostly)
    return gs_set == ext_set

In [5]:
def eval_intern(source, result_basis, exact) -> pd.DataFrame:
    result = result_basis.copy()
    if exact:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "Base" -> "A" -> "B" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit_exactly,          # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    else:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "Base" -> "A" -> "B" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)

    return result

def eval(source, exact) -> pd.DataFrame:

    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]], exact)

    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source, exact)

    return gs_eval, in_place



def print_eval(gs_eval) -> None:
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v:<4}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## YNORM

### Exact

In [6]:
ynorm_exact, ynorm_exact_extended = eval(answers, True)

print_eval(ynorm_exact)

### value ###
Base: True = 2073 || False = 135 || Quota = 93.89%
A   : True = 2100 || False = 108 || Quota = 95.11%

### unit ###
Base: True = 1983 || False = 225 || Quota = 89.81%
A   : True = 1967 || False = 241 || Quota = 89.09%



### ANY

In [7]:
ynorm_exact_any, ynorm_exact_extended_any = eval(answers, False)

print_eval(ynorm_exact_any)

### value ###
Base: True = 2090 || False = 118 || Quota = 94.66%
A   : True = 2108 || False = 100 || Quota = 95.47%

### unit ###
Base: True = 1991 || False = 217 || Quota = 90.17%
A   : True = 1967 || False = 241 || Quota = 89.09%



## Report-Level

In [8]:
cat = "value"  # oder "unit"
diff_mask = ynorm_exact_any[f"Base_{cat}_hit"] != ynorm_exact_any[f"A_{cat}_hit"]
diffs = ynorm_exact_any[diff_mask].copy()
diffs["Base_wins"] = diffs[f"Base_{cat}_hit"] & ~diffs[f"A_{cat}_hit"]
diffs["A_wins"]    = diffs[f"A_{cat}_hit"] & ~diffs[f"Base_{cat}_hit"]

summary = diffs.groupby("report_name")[["Base_wins", "A_wins"]].sum()
summary = summary[(summary["Base_wins"] > 0) | (summary["A_wins"] > 0)]
summary["net"] = summary["Base_wins"] - summary["A_wins"]
summary.sort_values("net", ascending=False)

,Base_wins,A_wins,net
report_name,,,
cardinal energy ltd_2021_report,6,0,6
inchcape plc_2022_report,2,0,2
salzgitter ag_2018_report,2,0,2
polaris_2021_report,1,0,1
Fresenius SE_2019_report,2,2,0
uniper_2019_report,2,4,-2
kfw_2018_report,0,3,-3
kureha corp_2020_report,0,3,-3
stantec inc_2019_report,0,3,-3
